## 세탁기 테이블 재구성

In [1]:
from pathlib import Path
import pandas as pd
import re

# =========================
# 경로 설정
# =========================
input_path = Path(r"C:/project_skn/project4/4th_project/products/data/preprocessing/washing_machine/lge_wm_all_products.csv")
output_path = Path("ProductWash.csv")

# =========================
# 변환 함수
# =========================
def blank_if_na(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def clean_int(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).replace(",", "")
    nums = re.findall(r"\d+(?:\.\d+)?", text)

    if not nums:
        return pd.NA

    return int(float(nums[0]))


def clean_proficiency(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).strip()

    if "해당없음" in text or "해당 없음" in text or "대상 외" in text or "비대상" in text:
        return pd.NA

    nums = re.findall(r"\d+", text)

    if not nums:
        return pd.NA

    return int(nums[0])


def split_size(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA, pd.NA, pd.NA

    text = str(value).replace(",", "").replace('"', "").strip()
    parts = re.split(r"\s*[xX×]\s*", text)

    if len(parts) < 3:
        return pd.NA, pd.NA, pd.NA

    width = clean_int(parts[0])
    height = clean_int(parts[1])
    depth = clean_int(parts[2])

    return width, height, depth


def clean_binary(value):
    if pd.isna(value) or str(value).strip() == "":
        return 0

    text = str(value).strip()

    no_keywords = ["없음", "제공안함", "해당없음", "해당 없음", "대상 외", "비대상", "미지원", "-"]

    if any(keyword in text for keyword in no_keywords):
        return 0

    if "있음" in text:
        return 1

    return 0


# =========================
# 원본 읽기
# =========================
df = pd.read_csv(input_path)

# =========================
# 크기 분리
# =========================
size_values = df["크기(mm)"].apply(split_size)

df["width(mm)"] = size_values.apply(lambda x: x[0])
df["height(mm)"] = size_values.apply(lambda x: x[1])
df["depth(mm)"] = size_values.apply(lambda x: x[2])

# =========================
# 새 DataFrame 구성
# =========================
new_df = pd.DataFrame({
    "product_code": [f"WMT{i:03d}" for i in range(len(df))],
    "name": df["제품명"].apply(blank_if_na),
    "img_link": df["이미지 링크"].apply(blank_if_na),
    "price": df["가격(원)"].apply(clean_int),
    "proficiency_level": df["에너지 소비효율등급"].apply(clean_proficiency),
    "power_consum(W)": df["소비전력(W)"].apply(clean_int),
    "washing_cap(kg)": df["세탁 용량(kg)"].apply(clean_int),
    "dryng_cap(kg)": df["건조 용량(kg)"].apply(clean_int),
    "width(mm)": df["width(mm)"],
    "height(mm)": df["height(mm)"],
    "depth(mm)": df["depth(mm)"],
    "weight(kg)": df["무게(kg)"].apply(clean_int),
    "color": df["색상"].apply(blank_if_na),
    "door_type": df["도어 디자인"].apply(blank_if_na),
    "control_type": df["조작 및 표현방식"].apply(blank_if_na),
    "water_temp": df["물온도"].apply(blank_if_na),
    "spin_op": df["탈수선택"].apply(clean_binary),
    "manual_link": df["제품 사용 설명서"].apply(blank_if_na)
})

# =========================
# 타입 지정
# =========================
str_cols = [
    "product_code", "name", "img_link",
    "color", "door_type", "control_type",
    "water_temp", "manual_link"
]

int_cols = [
    "price", "proficiency_level", "power_consum(W)",
    "washing_cap(kg)", "dryng_cap(kg)",
    "width(mm)", "height(mm)", "depth(mm)",
    "weight(kg)", "spin_op"
]

for col in str_cols:
    new_df[col] = new_df[col].astype("string")

for col in int_cols:
    new_df[col] = new_df[col].astype("Int64")

# =========================
# 저장
# =========================
new_df.to_csv(output_path, index=False, encoding="utf-8-sig", na_rep="")

print(new_df.head())
print(f"저장 완료: {output_path}")

  product_code       name                                           img_link  \
0       WMT000  LG 트롬 세탁기  https://www.lge.co.kr/kr/images/washing-machin...   
1       WMT001  LG 트롬 세탁기  https://www.lge.co.kr/kr/images/washing-machin...   
2       WMT002  LG 트롬 세탁기  https://www.lge.co.kr/kr/images/washing-machin...   
3       WMT003  LG 트롬 세탁기  https://www.lge.co.kr/kr/images/washing-machin...   
4       WMT004  LG 트롬 세탁기  https://www.lge.co.kr/kr/images/washing-machin...   

     price  proficiency_level  power_consum(W)  washing_cap(kg)  \
0  1150000                  1             2000               12   
1  1050000                  1             2000               12   
2  1250000                  1             2000               15   
3  1250000                  1             2000               17   
4  1200000                  3             2000               17   

   dryng_cap(kg)  width(mm)  height(mm)  depth(mm)  weight(kg)  \
0           <NA>        600         850        615

## door type -> door design 으로 바꾸고 html 다시 찾아서 넣기

In [3]:
from pathlib import Path
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from tqdm import tqdm


# =========================
# 경로 설정
# =========================
csv_path = Path("C:/project_skn/project4/4th_project/products/data/preprocessing/washing_machine/ProductWash.csv")          # 기존 결과 csv
link_txt_path = Path("C:/project_skn/project4/4th_project/products/data/preprocessing/washing_machine/washing_machine_product_links.txt") # 제품 링크가 줄바꿈으로 들어있는 txt
output_path = Path("C:/project_skn/project4/4th_project/products/data/preprocessing/washing_machine/ProductWash__update.csv")      # 그대로 덮어쓰기


# =========================
# 기존 CSV 읽기
# =========================
df = pd.read_csv(csv_path)

# door_type 컬럼명을 door_design으로 변경
if "door_type" in df.columns:
    df = df.rename(columns={"door_type": "door_design"})

# door_design 컬럼 없으면 생성
if "door_design" not in df.columns:
    df["door_design"] = ""


# =========================
# 링크 txt 읽기
# =========================
with open(link_txt_path, "r", encoding="utf-8") as f:
    links = [line.strip() for line in f if line.strip()]

print(f"CSV 행 개수: {len(df)}")
print(f"링크 개수: {len(links)}")


# =========================
# Selenium 설정
# =========================
options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

wait = WebDriverWait(driver, 10)


# =========================
# 함수
# =========================
def click_spec_more_button():
    """
    제품스펙 상세정보 펼치기 버튼 클릭
    """
    try:
        button = wait.until(
            EC.element_to_be_clickable((
                By.XPATH,
                "//a[@role='button' and contains(@title, '제품스펙 상세정보 펼치기')]"
            ))
        )
        driver.execute_script("arguments[0].click();", button)
        time.sleep(1)
    except:
        pass


def get_spec_value(spec_name):
    """
    dt 텍스트가 spec_name인 항목의 바로 다음 dd 값을 가져옴
    예: dt 도어 디자인 -> dd 글라스 도어
    """
    try:
        element = driver.find_element(
            By.XPATH,
            f"//dt[normalize-space()='{spec_name}']/following-sibling::dd[1]"
        )
        return element.text.strip()
    except:
        return ""


# =========================
# 링크 돌면서 door_design 다시 추출
# =========================
door_design_values = []

for link in tqdm(links, desc="door_design 수집중", ncols=120):

    try:
        driver.get(link)
        time.sleep(2)

        click_spec_more_button()

        door_design = get_spec_value("도어 디자인")
        door_design_values.append(door_design)

    except:
        door_design_values.append("")


driver.quit()


# =========================
# CSV에 값 반영
# 링크 순서와 CSV 행 순서가 같다고 가정
# =========================
for i, value in enumerate(door_design_values):
    if i < len(df):
        df.loc[i, "door_design"] = value


# =========================
# 저장
# =========================
df["door_design"] = df["door_design"].fillna("").astype("string")

df.to_csv(output_path, index=False, encoding="utf-8-sig", na_rep="")

print("저장 완료:", output_path)
print(df[["product_code", "name", "door_design"]].head())

CSV 행 개수: 156
링크 개수: 156


door_design 수집중: 100%|█████████████████████████████████████████████████████████████| 156/156 [48:27<00:00, 18.64s/it]


저장 완료: C:\project_skn\project4\4th_project\products\data\preprocessing\washing_machine\ProductWash__update.csv
  product_code       name door_design
0       WMT000  LG 트롬 세탁기      글라스 도어
1       WMT001  LG 트롬 세탁기      글라스 도어
2       WMT002  LG 트롬 세탁기      글라스 도어
3       WMT003  LG 트롬 세탁기      글라스 도어
4       WMT004  LG 트롬 세탁기      글라스 도어


## 타입 확인

In [9]:
import pandas as pd
from pathlib import Path

df = pd.read_csv("C:/project_skn/project4/4th_project/products/data/preprocessing/washing_machine/ProductWash__update.csv")      # 그대로 덮어쓰기

str_cols = [
    "product_code", "name", "img_link", "color",
    "door_design", "control_type", "water_temp", "manual_link"
]

int_cols = [
    "price", "proficiency_level", "power_consum(W)",
    "washing_cap(kg)", "dryng_cap(kg)",
    "width(mm)", "height(mm)", "depth(mm)",
    "weight(kg)", "spin_op"
]

for col in str_cols:
    df[col] = df[col].fillna("").astype("string")

for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

print(df.dtypes)

product_code         string[python]
name                 string[python]
img_link             string[python]
price                         Int64
proficiency_level             Int64
power_consum(W)               Int64
washing_cap(kg)               Int64
dryng_cap(kg)                 Int64
width(mm)                     Int64
height(mm)                    Int64
depth(mm)                     Int64
weight(kg)                    Int64
color                string[python]
door_design          string[python]
control_type         string[python]
water_temp           string[python]
spin_op                       Int64
manual_link          string[python]
dtype: object


In [5]:
from pathlib import Path
import pandas as pd

# =========================
# CSV 읽기
# =========================
csv_path = Path("C:/project_skn/project4/4th_project/products/data/preprocessing/washing_machine/ProductWash__update.csv")

df = pd.read_csv(csv_path)

# =========================
# 컬럼 타입 정의
# =========================

# str 타입 컬럼
str_cols = [
    "product_code",
    "name",
    "img_link",
    "color",
    "door_design",
    "control_type",
    "water_temp",
    "manual_link"
]

# int 타입 컬럼
int_cols = [
    "price",
    "proficiency_level",
    "power_consum(W)",
    "washing_cap(kg)",
    "dryng_cap(kg)",
    "width(mm)",
    "height(mm)",
    "depth(mm)",
    "weight(kg)",
    "spin_op"
]

# =========================
# 문자열 컬럼 변환
# =========================
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].fillna("").astype("string")

# =========================
# 정수 컬럼 변환
# 공란 허용 위해 Int64 사용
# =========================
for col in int_cols:
    if col in df.columns:

        # 숫자 변환
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        # Int64 타입 변환
        df[col] = df[col].astype("Int64")

# =========================
# 저장
# =========================
output_path = csv_path.parent / "type_fixed_ProductWash__update.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    na_rep=""
)

# =========================
# 결과 확인
# =========================
print(df.dtypes)

print(f"\n저장 완료: {output_path}")

product_code         string[python]
name                 string[python]
img_link             string[python]
price                         Int64
proficiency_level             Int64
power_consum(W)               Int64
washing_cap(kg)               Int64
dryng_cap(kg)                 Int64
width(mm)                     Int64
height(mm)                    Int64
depth(mm)                     Int64
weight(kg)                    Int64
color                string[python]
door_design          string[python]
control_type         string[python]
water_temp           string[python]
spin_op                       Int64
manual_link          string[python]
dtype: object

저장 완료: C:\project_skn\project4\4th_project\products\data\preprocessing\washing_machine\type_fixed_ProductWash__update.csv
